# 🫀 Aether-Pulse CDSS — XGBoost Training Notebook

**Clinical Decision Support System** — hybrid rule engine + XGBoost medication safety checker.

**Team:** Muhammad Hamza Shafoat, Tanisha Jain, Brian Liu, Arnav Dhiman, Jackson Raines, Basmala El-boudy  
**Mentor:** Ben Amoakoh

---

### ⚙️ Configuration
Set the variables in **Cell 0** before running the notebook.

| Variable | Description |
|----------|-------------|
| `DATA_SOURCE` | `"drive"` or `"huggingface"` |
| `DRIVE_DATA_PATH` | Path on Google Drive (used when `DATA_SOURCE="drive"`) |
| `HF_DATASET_NAME` | HuggingFace dataset identifier (used when `DATA_SOURCE="huggingface"`) |
| `LABEL_COL` | Target column name in the dataset |
| `OUTPUT_MODEL_PATH` | Where to save `model.pkl` |
| `TUNER` | `"grid"` or `"optuna"` |

In [ ]:
# ============================================================
# 🔧  USER CONFIGURATION — edit these variables before running
# ============================================================

DATA_SOURCE = "drive"          # "drive" | "huggingface"

# --- Google Drive settings (used when DATA_SOURCE="drive") ---
DRIVE_DATA_PATH = "/content/drive/MyDrive/aether_pulse/data/medication_safety.csv"

# --- HuggingFace settings (used when DATA_SOURCE="huggingface") ---
HF_DATASET_NAME = "your_org/aether_pulse_medication_safety"  # replace with real dataset
HF_SPLIT = "train"

# --- Training settings ---
LABEL_COL = "label"            # target column: 0=safe, 1=dangerous
TUNER = "grid"                 # "grid" | "optuna"
N_OPTUNA_TRIALS = 30           # only used when TUNER="optuna"

# --- Repository ---
REPO_URL = "https://github.com/arnavd371/Aether-Pulse"  # change if using a fork

# --- Output ---
OUTPUT_MODEL_PATH = "/content/drive/MyDrive/aether_pulse/model.pkl"
EVAL_OUTPUT_DIR = "/content/eval_outputs"

print("✅ Configuration loaded")

In [ ]:
# ============================================================
# Cell 1: Mount Google Drive / Load from HuggingFace
# ============================================================

import pandas as pd

if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    print(f"📂 Loading dataset from Drive: {DRIVE_DATA_PATH}")
    if DRIVE_DATA_PATH.endswith(".parquet"):
        df = pd.read_parquet(DRIVE_DATA_PATH)
    else:
        df = pd.read_csv(DRIVE_DATA_PATH)

elif DATA_SOURCE == "huggingface":
    # Install datasets library if not present
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
    from datasets import load_dataset as hf_load
    print(f"🤗 Loading dataset from HuggingFace: {HF_DATASET_NAME} (split={HF_SPLIT})")
    hf_dataset = hf_load(HF_DATASET_NAME, split=HF_SPLIT)
    df = hf_dataset.to_pandas()

else:
    raise ValueError(f"Unknown DATA_SOURCE: '{DATA_SOURCE}'. Choose 'drive' or 'huggingface'.")

print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Columns: {list(df.columns)}")
df.head()

In [ ]:
# ============================================================
# Cell 2: Install dependencies & import preprocessing pipeline
# ============================================================

import subprocess, sys

packages = [
    "xgboost", "scikit-learn", "pandas", "numpy",
    "fastapi", "uvicorn", "pydantic", "joblib",
    "optuna", "matplotlib", "seaborn", "pyyaml",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

# Clone repo if not already present (Colab session)
import os
if not os.path.exists("/content/Aether-Pulse"):
    subprocess.check_call(["git", "clone", REPO_URL, "/content/Aether-Pulse"])

sys.path.insert(0, "/content/Aether-Pulse/src")

import numpy as np
from ml_model.preprocessing import MedicationPreprocessor

preprocessor = MedicationPreprocessor()

y = df[LABEL_COL].astype(int).to_numpy()
X_raw = df.drop(columns=[LABEL_COL])
X = preprocessor.fit_transform(X_raw)

print(f"✅ Feature matrix shape: {X.shape}")
print(f"   Class balance — Safe: {(y==0).sum()}, Dangerous: {(y==1).sum()}")

In [ ]:
# ============================================================
# Cell 3: Train XGBoost model
# ============================================================

from ml_model.train import train

# Save the prepared dataframe to a temporary CSV so train() can load it
import tempfile, os

tmp_csv = "/tmp/aether_prepared_data.csv"
df.to_csv(tmp_csv, index=False)

train(
    data_path=tmp_csv,
    output_path=OUTPUT_MODEL_PATH,
    tuner=TUNER,
    n_trials=N_OPTUNA_TRIALS,
    label_col=LABEL_COL,
)

print(f"\n✅ Model saved to: {OUTPUT_MODEL_PATH}")

In [ ]:
# ============================================================
# Cell 4: Evaluate model and display metrics
# ============================================================

import os
from IPython.display import Image, display

from ml_model.evaluate import evaluate

os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

metrics = evaluate(
    data_path=tmp_csv,           # use same dataset for demo; replace with held-out test set
    model_path=OUTPUT_MODEL_PATH,
    label_col=LABEL_COL,
    output_dir=EVAL_OUTPUT_DIR,
    optimise_recall=True,
)

print("\n📊 Final Metrics:")
for k, v in metrics.items():
    print(f"   {k:12s}: {v:.4f}")

# Display plots inline
for plot_file in ["confusion_matrix.png", "roc_curve.png"]:
    path = os.path.join(EVAL_OUTPUT_DIR, plot_file)
    if os.path.isfile(path):
        display(Image(path))

In [ ]:
# ============================================================
# Cell 5: Save model to Drive (already done by train(), 
#         this cell verifies the file and copies eval plots)
# ============================================================

import shutil, os

model_size_mb = os.path.getsize(OUTPUT_MODEL_PATH) / 1e6
print(f"✅ Model file: {OUTPUT_MODEL_PATH} ({model_size_mb:.2f} MB)")

# Copy evaluation plots to Drive
drive_eval_dir = os.path.join(os.path.dirname(OUTPUT_MODEL_PATH), "eval_outputs")
if EVAL_OUTPUT_DIR != drive_eval_dir:
    os.makedirs(drive_eval_dir, exist_ok=True)
    for f in os.listdir(EVAL_OUTPUT_DIR):
        shutil.copy(os.path.join(EVAL_OUTPUT_DIR, f), drive_eval_dir)
    print(f"📂 Evaluation plots saved to: {drive_eval_dir}")

print("\n🎉 Aether-Pulse training pipeline complete!")